<a href="https://colab.research.google.com/github/Miranita-ar/Skripsi-Gojek-App-Review/blob/main/Code/(XD)_Skripsi_IndoBERT_LIME_Analysisi.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Local Intepretable Model-agnostic Explanation

# BAB 1. PERSIAPAN LINGKUNGAN

## 1.1 Install Library

In [ ]:
# =====================================================
# CELL 1 : INSTALL LIBRARY
# =====================================================

print("="*60)
print("INSTALL LIBRARY")
print("="*60)

!pip -q install lime
!pip -q install transformers
!pip -q install sentencepiece
!pip -q install scikit-learn

print()

print("Seluruh library berhasil diinstall.")

INSTALL LIBRARY
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 275.7/275.7 kB 13.0 MB/s eta 0:00:00
  Preparing metadata (setup.py) ... done

Seluruh library berhasil diinstall.


## 1.2 Import Library

In [ ]:
# =====================================================
# CELL 2 : IMPORT LIBRARY
# =====================================================

# =====================================================
# SYSTEM
# =====================================================

import os
import gc
import json
import random
import warnings

warnings.filterwarnings("ignore")

# =====================================================
# NUMERICAL
# =====================================================

import numpy as np
import pandas as pd

# =====================================================
# VISUALIZATION
# =====================================================

import matplotlib.pyplot as plt

# =====================================================
# PROGRESS BAR
# =====================================================

from tqdm.auto import tqdm

# =====================================================
# PYTORCH
# =====================================================

import torch

# =====================================================
# TRANSFORMERS
# =====================================================

from transformers import (
    AutoTokenizer,
    AutoModelForSequenceClassification
)

# =====================================================
# LIME
# =====================================================

from lime.lime_text import LimeTextExplainer

# =====================================================
# DISPLAY
# =====================================================

from IPython.display import display

print("="*60)
print("IMPORT LIBRARY")
print("="*60)

print()

print("Semua library berhasil diimport.")

IMPORT LIBRARY

Semua library berhasil diimport.


## 1.3 Mount Google Drive

In [ ]:
# =====================================================
# CELL 3 : MOUNT GOOGLE DRIVE
# =====================================================

from google.colab import drive

print("="*60)
print("MOUNT GOOGLE DRIVE")
print("="*60)

drive.mount("/content/drive")

print()

print("Google Drive berhasil dihubungkan.")

MOUNT GOOGLE DRIVE
Mounted at /content/drive

Google Drive berhasil dihubungkan.


## 1.4 Konfigurasi Global

In [ ]:
# =====================================================
# CELL 4 : KONFIGURASI GLOBAL
# =====================================================

print("="*60)
print("KONFIGURASI GLOBAL")
print("="*60)

# =====================================================
# RANDOM SEED
# =====================================================

SEED = 42

random.seed(SEED)
np.random.seed(SEED)

torch.manual_seed(SEED)

if torch.cuda.is_available():

    torch.cuda.manual_seed_all(SEED)

# =====================================================
# DEVICE
# =====================================================

DEVICE = torch.device(

    "cuda"

    if torch.cuda.is_available()

    else

    "cpu"

)

print()

print(f"Random Seed : {SEED}")

print(f"Device      : {DEVICE}")

if DEVICE.type == "cuda":

    print(

        f"GPU         : {torch.cuda.get_device_name(0)}"

    )

print()

print("Konfigurasi Global berhasil dibuat.")

KONFIGURASI GLOBAL

Random Seed : 42
Device      : cuda
GPU         : Tesla T4

Konfigurasi Global berhasil dibuat.


## 1.5 Utility Function

In [ ]:
# =====================================================
# CELL 5 : UTILITY FUNCTION
# =====================================================

def print_header(text):

    print()

    print("="*60)

    print(text)

    print("="*60)


def print_success(text):

    print(f"✅ {text}")


def print_info(text):

    print(f"ℹ️ {text}")


def print_warning(text):

    print(f"⚠️ {text}")


def count_files(folder, extension):

    if not os.path.exists(folder):

        return 0

    return len(

        [

            f

            for f in os.listdir(folder)

            if f.endswith(extension)

        ]

    )

print_header("UTILITY FUNCTION")

print()

print_success(

    "Utility Function berhasil dibuat."

)


UTILITY FUNCTION

✅ Utility Function berhasil dibuat.


## 1.6 Konfigurasi Path

In [ ]:
# =====================================================
# CELL 6 : KONFIGURASI PATH
# =====================================================

PATHS = {

    # Root Project
    "PROJECT":
        "/content/drive/MyDrive/Skripsi_IndoBERT",

    # Model Final
    "MODEL":
        "/content/drive/MyDrive/Skripsi_IndoBERT/model_indobert_terbaik",

    # Notebook XB
    "XB":
        "/content/drive/MyDrive/Skripsi_IndoBERT/xai/XB_Persiapan_XAI",

    "XB_CSV":
        "/content/drive/MyDrive/Skripsi_IndoBERT/xai/XB_Persiapan_XAI/csv",

    "XB_LOG":
        "/content/drive/MyDrive/Skripsi_IndoBERT/xai/XB_Persiapan_XAI/log",

    # Notebook XD
    "XD":
        "/content/drive/MyDrive/Skripsi_IndoBERT/xai/XD_LIME_Analysis"

}

print_header("KONFIGURASI PATH")

print()

for key, value in PATHS.items():

    print_info(

        f"{key:<8}: {value}"

    )

print()

print_success(

    "Seluruh path berhasil dikonfigurasi."

)


KONFIGURASI PATH

ℹ️ PROJECT : /content/drive/MyDrive/Skripsi_IndoBERT
ℹ️ MODEL   : /content/drive/MyDrive/Skripsi_IndoBERT/model_indobert_terbaik
ℹ️ XB      : /content/drive/MyDrive/Skripsi_IndoBERT/xai/XB_Persiapan_XAI
ℹ️ XB_CSV  : /content/drive/MyDrive/Skripsi_IndoBERT/xai/XB_Persiapan_XAI/csv
ℹ️ XB_LOG  : /content/drive/MyDrive/Skripsi_IndoBERT/xai/XB_Persiapan_XAI/log
ℹ️ XD      : /content/drive/MyDrive/Skripsi_IndoBERT/xai/XD_LIME_Analysis

✅ Seluruh path berhasil dikonfigurasi.


## 1.7 Membuat Struktur Folder

In [ ]:
# =====================================================
# CELL 7 : CREATE DIRECTORY
# =====================================================

LOCAL_DIR = os.path.join(

    PATHS["XD"],

    "local"

)

HTML_DIR = os.path.join(

    LOCAL_DIR,

    "html"

)

PNG_DIR = os.path.join(

    LOCAL_DIR,

    "png"

)

METADATA_DIR = os.path.join(

    LOCAL_DIR,

    "metadata"

)

GLOBAL_DIR = os.path.join(

    PATHS["XD"],

    "global"

)

CSV_DIR = os.path.join(

    PATHS["XD"],

    "csv"

)

README_DIR = os.path.join(

    PATHS["XD"],

    "readme"

)

for folder in [

    PATHS["XD"],

    LOCAL_DIR,

    HTML_DIR,

    PNG_DIR,

    METADATA_DIR,

    GLOBAL_DIR,

    CSV_DIR,

    README_DIR

]:

    os.makedirs(

        folder,

        exist_ok=True

    )

print_header("CREATE DIRECTORY")

print()

print_success(

    "Seluruh folder output berhasil dibuat."

)


CREATE DIRECTORY

✅ Seluruh folder output berhasil dibuat.


## 1.8 Validasi Struktur Folder

In [ ]:
# =====================================================
# CELL 8 : VALIDASI DIRECTORY
# =====================================================

folders = [

    PATHS["XD"],

    LOCAL_DIR,

    HTML_DIR,

    PNG_DIR,

    METADATA_DIR,

    GLOBAL_DIR,

    CSV_DIR,

    README_DIR

]

validation = pd.DataFrame({

    "Folder": folders,

    "Status":

        [

            os.path.exists(folder)

            for folder in folders

        ]

})

display(validation)

print()

print_info(

    f"Jumlah Folder : {len(validation)}"

)

print()

if validation["Status"].all():

    print_success(

        "Seluruh folder berhasil divalidasi."

    )

else:

    print_warning(

        "Masih terdapat folder yang belum dibuat."

    )

,Folder,Status
0,/content/drive/MyDrive/Skripsi_IndoBERT/xai/XD...,True
1,/content/drive/MyDrive/Skripsi_IndoBERT/xai/XD...,True
2,/content/drive/MyDrive/Skripsi_IndoBERT/xai/XD...,True
3,/content/drive/MyDrive/Skripsi_IndoBERT/xai/XD...,True
4,/content/drive/MyDrive/Skripsi_IndoBERT/xai/XD...,True
5,/content/drive/MyDrive/Skripsi_IndoBERT/xai/XD...,True
6,/content/drive/MyDrive/Skripsi_IndoBERT/xai/XD...,True
7,/content/drive/MyDrive/Skripsi_IndoBERT/xai/XD...,True



ℹ️ Jumlah Folder : 8

✅ Seluruh folder berhasil divalidasi.


# 2. PERSIAPAN MODEL DAN DATASET

## 2.1 Load Tokenizer

In [ ]:
# =====================================================
# CELL 9 : LOAD TOKENIZER
# =====================================================

print_header("LOAD TOKENIZER")

print()

print_info(

    f"Model : {PATHS['MODEL']}"

)

tokenizer = AutoTokenizer.from_pretrained(

    PATHS["MODEL"]

)

print()

print_success(

    "Tokenizer berhasil dimuat."

)


LOAD TOKENIZER

ℹ️ Model : /content/drive/MyDrive/Skripsi_IndoBERT/model_indobert_terbaik

✅ Tokenizer berhasil dimuat.


## 2.2 Load Model

In [ ]:
# =====================================================
# CELL 10 : LOAD MODEL
# =====================================================

print_header("LOAD MODEL")

print()

print_info(

    f"Model : {PATHS['MODEL']}"

)

model = AutoModelForSequenceClassification.from_pretrained(

    PATHS["MODEL"]

)

model.to(

    DEVICE

)

model.eval()

print()

print_success(

    "Model berhasil dimuat."

)


LOAD MODEL

ℹ️ Model : /content/drive/MyDrive/Skripsi_IndoBERT/model_indobert_terbaik


Loading weights:   0%|          | 0/201 [00:00<?, ?it/s]


✅ Model berhasil dimuat.


## 2.3 Validasi Model

In [ ]:
# =====================================================
# CELL 11 : VALIDASI MODEL
# =====================================================

print_header("VALIDASI MODEL")

validation = {

    "Tokenizer":

        tokenizer is not None,

    "Model":

        model is not None,

    "Device":

        DEVICE.type

}

validation_df = pd.DataFrame({

    "Komponen":

        validation.keys(),

    "Nilai":

        validation.values()

})

display(

    validation_df

)

print()

print_success(

    "Model berhasil divalidasi."

)


VALIDASI MODEL


,Komponen,Nilai
0,Tokenizer,True
1,Model,True
2,Device,cuda



✅ Model berhasil divalidasi.


## 2.4 Load Dataset

In [ ]:
# =====================================================
# CELL 12 : LOAD DATASET
# =====================================================

SELECTED_DATA_PATH = os.path.join(

    PATHS["XB_CSV"],

    "selected_reviews_xai.csv"

)

print_header("LOAD DATASET")

print()

print_info(

    f"Dataset : {SELECTED_DATA_PATH}"

)

if not os.path.exists(

    SELECTED_DATA_PATH

):

    raise FileNotFoundError(

        SELECTED_DATA_PATH

    )

selected_reviews_df = pd.read_csv(

    SELECTED_DATA_PATH

)

print()

print_success(

    "Dataset berhasil dimuat."

)

print_info(

    f"Jumlah Sampel : {len(selected_reviews_df)}"

)


LOAD DATASET

ℹ️ Dataset : /content/drive/MyDrive/Skripsi_IndoBERT/xai/XB_Persiapan_XAI/csv/selected_reviews_xai.csv

✅ Dataset berhasil dimuat.
ℹ️ Jumlah Sampel : 30


## 2.5 Validasi Dataset

In [ ]:
# =====================================================
# CELL 13 : VALIDASI DATASET
# =====================================================

print_header("VALIDASI DATASET")

display(

    selected_reviews_df.head()

)

print()

print_info(

    f"Jumlah Baris : {selected_reviews_df.shape[0]}"

)

print_info(

    f"Jumlah Kolom : {selected_reviews_df.shape[1]}"

)

print()

print_info(

    "Daftar Kolom Dataset"

)

display(

    pd.DataFrame({

        "Kolom":

            selected_reviews_df.columns

    })

)

print()

print_success(

    "Dataset berhasil divalidasi."

)


VALIDASI DATASET


,analysis_order,sample_id,text,actual_label,actual_sentiment,predicted_label,predicted_sentiment,prob_negatif,prob_netral,prob_positif,confidence,correct,prediction_case,confidence_level,selection_reason,sample_group,explain_status
0,1,XAI_001,sangat membantu dan mudah,2,Positif,2,Positif,0.000885,0.000943,0.998172,0.998172,True,Benar_Positif,Tinggi,Prediksi benar kelas Positif,High Confidence,Pending
1,2,XAI_002,mudah dan sangat membantu sekali,2,Positif,2,Positif,0.000879,0.000953,0.998167,0.998167,True,Benar_Positif,Tinggi,Prediksi benar kelas Positif,High Confidence,Pending
2,3,XAI_003,sangat membantu sekali,2,Positif,2,Positif,0.000898,0.000940,0.998162,0.998162,True,Benar_Positif,Tinggi,Prediksi benar kelas Positif,High Confidence,Pending
3,4,XAI_004,baik dan sangat membantu,2,Positif,2,Positif,0.000917,0.000924,0.998159,0.998159,True,Benar_Positif,Tinggi,Prediksi benar kelas Positif,High Confidence,Pending
4,5,XAI_005,sangat membantu sekali dalam segala aktivitas,2,Positif,2,Positif,0.000891,0.000954,0.998155,0.998155,True,Benar_Positif,Tinggi,Prediksi benar kelas Positif,High Confidence,Pending



ℹ️ Jumlah Baris : 30
ℹ️ Jumlah Kolom : 17

ℹ️ Daftar Kolom Dataset


,Kolom
0,analysis_order
1,sample_id
2,text
3,actual_label
4,actual_sentiment
5,predicted_label
6,predicted_sentiment
7,prob_negatif
8,prob_netral
9,prob_positif



✅ Dataset berhasil divalidasi.


## 2.6 Distribusi Label

In [ ]:
# =====================================================
# CELL 14 : DISTRIBUSI LABEL
# =====================================================

print_header("DISTRIBUSI LABEL")

actual_distribution = (

    selected_reviews_df["actual_sentiment"]

    .value_counts()

    .sort_index()

)

predicted_distribution = (

    selected_reviews_df["predicted_sentiment"]

    .value_counts()

    .sort_index()

)

distribution_df = pd.DataFrame({

    "Actual":

        actual_distribution,

    "Predicted":

        predicted_distribution

})

display(

    distribution_df

)

print()

print_success(

    "Distribusi Label berhasil dibuat."

)


DISTRIBUSI LABEL


,Actual,Predicted
Negatif,7,12
Netral,17,7
Positif,6,11



✅ Distribusi Label berhasil dibuat.


## 2.7 Validasi Akhir BAB 2

In [ ]:
# =====================================================
# CELL 15 : VALIDASI AKHIR BAB 2
# =====================================================

print_header("VALIDASI AKHIR BAB 2")

validation = {

    "Tokenizer":

        tokenizer is not None,

    "Model":

        model is not None,

    "Dataset":

        len(selected_reviews_df),

    "Jumlah Label":

        model.config.num_labels,

    "Jumlah Sampel":

        len(selected_reviews_df)

}

validation_df = pd.DataFrame({

    "Komponen":

        validation.keys(),

    "Nilai":

        validation.values()

})

display(

    validation_df

)

print()

print_success(

    "BAB 2 berhasil diselesaikan."

)


VALIDASI AKHIR BAB 2


,Komponen,Nilai
0,Tokenizer,True
1,Model,True
2,Dataset,30
3,Jumlah Label,3
4,Jumlah Sampel,30



✅ BAB 2 berhasil diselesaikan.


# 3. PERSIAPAN LIME EXPLAINER

## 3.1 Membuat Prediction Function

In [ ]:
# =====================================================
# CELL 16 : CREATE PREDICTION FUNCTION
# =====================================================

print_header("CREATE PREDICTION FUNCTION")

id2label = {

    0: "Negatif",

    1: "Netral",

    2: "Positif"

}

label2id = {

    v: k

    for k, v in id2label.items()

}

BATCH_SIZE = 32


def predict_proba(texts):

    """
    Prediction Function khusus LIME
    Menggunakan mini-batch agar GPU tidak OutOfMemory.
    """

    probabilities = []

    model.eval()

    for start in range(0, len(texts), BATCH_SIZE):

        batch = texts[start:start+BATCH_SIZE]

        encoded = tokenizer(

            batch,

            padding=True,

            truncation=True,

            max_length=512,

            return_tensors="pt"

        )

        encoded = {

            k: v.to(DEVICE)

            for k, v in encoded.items()

        }

        with torch.no_grad():

            outputs = model(

                **encoded

            )

            probs = torch.softmax(

                outputs.logits,

                dim=1

            )

        probabilities.append(

            probs.cpu().numpy()

        )

        del encoded

        del outputs

        del probs

        torch.cuda.empty_cache()

    probabilities = np.vstack(

        probabilities

    )

    return probabilities


print()

print_success(

    "Prediction Function berhasil dibuat."

)


CREATE PREDICTION FUNCTION

✅ Prediction Function berhasil dibuat.


## 3.2 Validasi Prediction Function

In [ ]:
# =====================================================
# CELL 17 : VALIDASI PREDICTION FUNCTION
# =====================================================

print_header("VALIDASI PREDICTION FUNCTION")

sample_prediction = predict_proba(

    [

        "aplikasinya sangat membantu"

    ]

)

prediction_df = pd.DataFrame(

    sample_prediction,

    columns=[

        "Negatif",

        "Netral",

        "Positif"

    ]

)

display(

    prediction_df

)

print()

print_info(

    f"Shape : {sample_prediction.shape}"

)

print()

if sample_prediction.shape[1] == 3:

    print_success(

        "Prediction Function berhasil divalidasi."

    )

else:

    print_warning(

        "Prediction Function tidak sesuai."

    )


VALIDASI PREDICTION FUNCTION


,Negatif,Netral,Positif
0,0.000858,0.001013,0.998129



ℹ️ Shape : (1, 3)

✅ Prediction Function berhasil divalidasi.


## 3.3 Inisialisasi LIME Explainer

In [ ]:
# =====================================================
# CELL 18 : INITIALIZE LIME EXPLAINER
# =====================================================

print_header("INITIALIZE LIME EXPLAINER")

lime_explainer = LimeTextExplainer(

    class_names=[

        "Negatif",

        "Netral",

        "Positif"

    ],

    random_state=SEED

)

print()

print_success(

    "LIME Explainer berhasil dibuat."

)


INITIALIZE LIME EXPLAINER

✅ LIME Explainer berhasil dibuat.


## 3.4 Validasi LIME Explainer

In [ ]:
# =====================================================
# CELL 19 : VALIDASI LIME EXPLAINER
# =====================================================

print_header("VALIDASI LIME EXPLAINER")

validation = {

    "Explainer":

        lime_explainer is not None,

    "Prediction Function":

        callable(predict_proba)

}

validation_df = pd.DataFrame({

    "Komponen":

        validation.keys(),

    "Status":

        validation.values()

})

display(

    validation_df

)

print()

print_success(

    "LIME Explainer berhasil divalidasi."

)


VALIDASI LIME EXPLAINER


,Komponen,Status
0,Explainer,True
1,Prediction Function,True



✅ LIME Explainer berhasil divalidasi.


## 3.5 Konfigurasi Explainability

In [ ]:
# =====================================================
# CELL 20 : KONFIGURASI EXPLAINABILITY
# =====================================================

print_header("KONFIGURASI EXPLAINABILITY")

NUM_FEATURES = 15

NUM_SAMPLES = 1000

LIME_RANDOM_STATE = SEED

print()

print_info(

    f"Jumlah Feature : {NUM_FEATURES}"

)

print_info(

    f"Jumlah Sampling : {NUM_SAMPLES}"

)

print_info(

    f"Batch Size : {BATCH_SIZE}"

)

print_info(

    f"Random State : {LIME_RANDOM_STATE}"

)

print()

print_success(

    "Konfigurasi Explainability berhasil dibuat."

)


KONFIGURASI EXPLAINABILITY

ℹ️ Jumlah Feature : 15
ℹ️ Jumlah Sampling : 1000
ℹ️ Batch Size : 32
ℹ️ Random State : 42

✅ Konfigurasi Explainability berhasil dibuat.


## 3.6 Validasi Konfigurasi Explainability

In [ ]:
# =====================================================
# CELL 21 : VALIDASI KONFIGURASI EXPLAINABILITY
# =====================================================

print_header("VALIDASI KONFIGURASI EXPLAINABILITY")

validation = {

    "Prediction Function":

        callable(predict_proba),

    "LIME Explainer":

        lime_explainer is not None,

    "Number Feature":

        NUM_FEATURES,

    "Number Sampling":

        NUM_SAMPLES,

    "Random State":

        LIME_RANDOM_STATE

}

validation_df = pd.DataFrame({

    "Komponen":

        validation.keys(),

    "Nilai":

        validation.values()

})

display(

    validation_df

)

print()

print_success(

    "Konfigurasi Explainability berhasil divalidasi."

)


VALIDASI KONFIGURASI EXPLAINABILITY


,Komponen,Nilai
0,Prediction Function,True
1,LIME Explainer,True
2,Number Feature,15
3,Number Sampling,1000
4,Random State,42



✅ Konfigurasi Explainability berhasil divalidasi.


## 3.7 Persiapan Output

In [ ]:
# =====================================================
# CELL 22 : PERSIAPAN OUTPUT
# =====================================================

print_header("PERSIAPAN OUTPUT")

LIME_RESULTS = {}

print()

print_info(

    f"Jumlah Sampel : {len(selected_reviews_df)}"

)

print()

print_success(

    "Dictionary hasil LIME berhasil dipersiapkan."

)


PERSIAPAN OUTPUT

ℹ️ Jumlah Sampel : 30

✅ Dictionary hasil LIME berhasil dipersiapkan.


## 3.8 Validasi Akhir BAB 3

In [ ]:
# =====================================================
# CELL 23 : VALIDASI AKHIR BAB 3
# =====================================================

print_header("VALIDASI AKHIR BAB 3")

validation = {

    "Prediction Function":

        callable(predict_proba),

    "LIME Explainer":

        lime_explainer is not None,

    "Dictionary":

        isinstance(

            LIME_RESULTS,

            dict

        ),

    "Jumlah Sampel":

        len(selected_reviews_df)

}

validation_df = pd.DataFrame({

    "Komponen":

        validation.keys(),

    "Nilai":

        validation.values()

})

display(

    validation_df

)

print()

print_success(

    "BAB 3 berhasil diselesaikan."

)


VALIDASI AKHIR BAB 3


,Komponen,Nilai
0,Prediction Function,True
1,LIME Explainer,True
2,Dictionary,True
3,Jumlah Sampel,30



✅ BAB 3 berhasil diselesaikan.


# 4. GENERATE LIME EXPLANATION

## 4.1 Generate LIME Explanation

In [ ]:
# =====================================================
# CELL 24 : GENERATE LIME EXPLANATION
# =====================================================

print_header("GENERATE LIME EXPLANATION")

print()

print_info(

    f"Total Sampel : {len(selected_reviews_df)}"

)

print_info(

    f"Jumlah Feature : {NUM_FEATURES}"

)

print_info(

    f"Sampling LIME : {NUM_SAMPLES}"

)

print()

print_success(

    "Notebook siap melakukan proses Generate LIME Explanation."

)


GENERATE LIME EXPLANATION

ℹ️ Total Sampel : 30
ℹ️ Jumlah Feature : 15
ℹ️ Sampling LIME : 1000

✅ Notebook siap melakukan proses Generate LIME Explanation.


## 4.2 Generate LIME Explanation

In [ ]:
# =====================================================
# CELL 25 : GENERATE LIME EXPLANATION
# =====================================================

print_header("GENERATE LIME EXPLANATION")

import gc

LIME_RESULTS = {}

success = 0

for _, row in tqdm(

    selected_reviews_df.iterrows(),

    total=len(selected_reviews_df),

    desc="Generate LIME"

):

    sample_id = row["sample_id"]

    text = row["text"]

    actual_label = int(row["actual_label"])

    predicted_label = int(row["predicted_label"])

    predicted_sentiment = row["predicted_sentiment"]

    probabilities = predict_proba(

        [text]

    )[0]

    explanation = lime_explainer.explain_instance(

        text_instance=text,

        classifier_fn=predict_proba,

        labels=[predicted_label],

        num_features=NUM_FEATURES,

        num_samples=NUM_SAMPLES

    )

    LIME_RESULTS[sample_id] = {

        "sample_id":

            sample_id,

        "text":

            text,

        "actual_label":

            actual_label,

        "predicted_label":

            predicted_label,

        "predicted_sentiment":

            predicted_sentiment,

        "probabilities":

            probabilities,

        "lime_explanation":

            explanation

    }

    success += 1

    gc.collect()

    torch.cuda.empty_cache()

print()

print_info(

    f"Explanation berhasil : {success}"

)

print()

print_success(

    "Generate LIME Explanation selesai."

)


GENERATE LIME EXPLANATION


Generate LIME:   0%|          | 0/30 [00:00<?, ?it/s]


ℹ️ Explanation berhasil : 30

✅ Generate LIME Explanation selesai.


## 4.3 Validasi LIME Explanation

In [ ]:
# =====================================================
# CELL 26 : VALIDASI LIME EXPLANATION
# =====================================================

print_header("VALIDASI LIME EXPLANATION")

expected = len(selected_reviews_df)

obtained = len(LIME_RESULTS)

print()

print_info(

    f"Expected : {expected}"

)

print_info(

    f"Obtained : {obtained}"

)

print()

if expected == obtained:

    print_success(

        "Seluruh LIME Explanation berhasil dibuat."

    )

else:

    print_warning(

        "Masih terdapat explanation yang gagal."

    )


VALIDASI LIME EXPLANATION

ℹ️ Expected : 30
ℹ️ Obtained : 30

✅ Seluruh LIME Explanation berhasil dibuat.


## 4.4 Generate Feature Importance

In [ ]:
# =====================================================
# CELL 27 : GENERATE FEATURE IMPORTANCE
# =====================================================

print_header("GENERATE FEATURE IMPORTANCE")

feature_success = 0

for sample_id, sample_data in tqdm(

    LIME_RESULTS.items(),

    desc="Generate Feature"

):

    explanation = sample_data["lime_explanation"]

    predicted_label = sample_data["predicted_label"]

    feature_list = explanation.as_list(

        label=predicted_label

    )

    feature_df = pd.DataFrame(

        feature_list,

        columns=[

            "Token",

            "Weight"

        ]

    )

    feature_df["|Weight|"] = (

        feature_df["Weight"]

        .abs()

    )

    feature_df["Direction"] = np.where(

        feature_df["Weight"] >= 0,

        "Positif",

        "Negatif"

    )

    feature_df = feature_df.sort_values(

        "|Weight|",

        ascending=False

    ).reset_index(drop=True)

    sample_data["feature_statistics"] = feature_df

    feature_success += 1

print()

print_info(

    f"Feature berhasil : {feature_success}"

)

print()

print_success(

    "Feature Importance berhasil dibuat."

)


GENERATE FEATURE IMPORTANCE


Generate Feature:   0%|          | 0/30 [00:00<?, ?it/s]


ℹ️ Feature berhasil : 30

✅ Feature Importance berhasil dibuat.


## 4.5 Validasi Feature Importance

In [ ]:
# =====================================================
# CELL 28 : VALIDASI FEATURE IMPORTANCE
# =====================================================

print_header("VALIDASI FEATURE IMPORTANCE")

expected = len(LIME_RESULTS)

obtained = sum(

    "feature_statistics" in v

    for v in LIME_RESULTS.values()

)

print()

print_info(

    f"Expected : {expected}"

)

print_info(

    f"Obtained : {obtained}"

)

print()

if expected == obtained:

    print_success(

        "Seluruh Feature Importance berhasil dibuat."

    )

else:

    print_warning(

        "Masih terdapat Feature Importance yang gagal."

    )


VALIDASI FEATURE IMPORTANCE

ℹ️ Expected : 30
ℹ️ Obtained : 30

✅ Seluruh Feature Importance berhasil dibuat.


## 4.6 Generate Metadata

In [ ]:
# =====================================================
# CELL 29 : GENERATE METADATA
# =====================================================

print_header("GENERATE METADATA")

metadata_success = 0

for sample_id, sample_data in tqdm(

    LIME_RESULTS.items(),

    desc="Generate Metadata"

):

    feature_df = sample_data["feature_statistics"]

    positive_df = feature_df[

        feature_df["Weight"] > 0

    ]

    negative_df = feature_df[

        feature_df["Weight"] < 0

    ]

    sample_data["metadata"] = {

        "feature_count":

            len(feature_df),

        "top_feature":

            feature_df.iloc[0]["Token"],

        "top_positive":

            positive_df.iloc[0]["Token"]

            if len(positive_df) > 0

            else None,

        "top_negative":

            negative_df.iloc[0]["Token"]

            if len(negative_df) > 0

            else None,

        "max_weight":

            float(

                feature_df["|Weight|"].max()

            ),

        "mean_weight":

            float(

                feature_df["|Weight|"].mean()

            )

    }

    metadata_success += 1

print()

print_info(

    f"Metadata berhasil : {metadata_success}"

)

print()

print_success(

    "Metadata berhasil dibuat."

)


GENERATE METADATA


Generate Metadata:   0%|          | 0/30 [00:00<?, ?it/s]


ℹ️ Metadata berhasil : 30

✅ Metadata berhasil dibuat.


## 4.7 Validasi Metadata

In [ ]:
# =====================================================
# CELL 30 : VALIDASI METADATA
# =====================================================

print_header("VALIDASI METADATA")

expected = len(LIME_RESULTS)

obtained = sum(

    "metadata" in v

    for v in LIME_RESULTS.values()

)

print()

print_info(

    f"Expected : {expected}"

)

print_info(

    f"Obtained : {obtained}"

)

print()

if expected == obtained:

    print_success(

        "Seluruh Metadata berhasil dibuat."

    )

else:

    print_warning(

        "Masih terdapat Metadata yang gagal."

    )


VALIDASI METADATA

ℹ️ Expected : 30
ℹ️ Obtained : 30

✅ Seluruh Metadata berhasil dibuat.


# 5. LIME LOCAL EXPLANATION

## 5.1 Persiapan Folder Output

In [ ]:
# =====================================================
# CELL 31 : PERSIAPAN FOLDER OUTPUT
# =====================================================

print_header("PERSIAPAN FOLDER OUTPUT")

ROOT_OUTPUT_DIR = os.path.join(

    PATHS["PROJECT"],

    "xai",

    "XD_LIME_Analysis"

)

LOCAL_HTML_DIR = os.path.join(

    ROOT_OUTPUT_DIR,

    "local",

    "html"

)

LOCAL_PNG_DIR = os.path.join(

    ROOT_OUTPUT_DIR,

    "local",

    "png"

)

LOCAL_CSV_DIR = os.path.join(

    ROOT_OUTPUT_DIR,

    "local",

    "csv"

)

LOCAL_METADATA_DIR = os.path.join(

    ROOT_OUTPUT_DIR,

    "local",

    "metadata"

)

README_DIR = os.path.join(

    ROOT_OUTPUT_DIR,

    "readme"

)

GLOBAL_DIR = os.path.join(

    ROOT_OUTPUT_DIR,

    "global"

)

CSV_DIR = os.path.join(

    ROOT_OUTPUT_DIR,

    "csv"

)

folders = [

    ROOT_OUTPUT_DIR,

    LOCAL_HTML_DIR,

    LOCAL_PNG_DIR,

    LOCAL_CSV_DIR,

    LOCAL_METADATA_DIR,

    README_DIR,

    GLOBAL_DIR,

    CSV_DIR

]

for folder in folders:

    os.makedirs(

        folder,

        exist_ok=True

    )

print()

print_info(

    f"Folder dibuat : {len(folders)}"

)

print()

print_success(

    "Seluruh folder output berhasil dibuat."

)


PERSIAPAN FOLDER OUTPUT

ℹ️ Folder dibuat : 8

✅ Seluruh folder output berhasil dibuat.


## 5.2 Validasi Folder Output

In [ ]:
# =====================================================
# CELL 32 : VALIDASI FOLDER OUTPUT
# =====================================================

print_header("VALIDASI FOLDER OUTPUT")

validation = []

for folder in folders:

    validation.append({

        "Folder":

            os.path.basename(folder),

        "Exists":

            os.path.exists(folder)

    })

validation_df = pd.DataFrame(

    validation

)

display(

    validation_df

)

print()

print_success(

    "Seluruh folder output berhasil divalidasi."

)


VALIDASI FOLDER OUTPUT


,Folder,Exists
0,XD_LIME_Analysis,True
1,html,True
2,png,True
3,csv,True
4,metadata,True
5,readme,True
6,global,True
7,csv,True



✅ Seluruh folder output berhasil divalidasi.


## 5.3 Generate HTML Explanation

In [ ]:
# =====================================================
# CELL 33 : GENERATE HTML EXPLANATION
# =====================================================

print_header("GENERATE HTML EXPLANATION")

html_success = 0

for sample_id, sample_data in tqdm(

    LIME_RESULTS.items(),

    desc="Generate HTML"

):

    explanation = sample_data["lime_explanation"]

    html_path = os.path.join(

        LOCAL_HTML_DIR,

        f"{sample_id}.html"

    )

    explanation.save_to_file(

        html_path

    )

    sample_data["html_path"] = html_path

    html_success += 1

print()

print_info(

    f"HTML berhasil : {html_success}"

)

print()

print_success(

    "Seluruh HTML berhasil dibuat."

)


GENERATE HTML EXPLANATION


Generate HTML:   0%|          | 0/30 [00:00<?, ?it/s]


ℹ️ HTML berhasil : 30

✅ Seluruh HTML berhasil dibuat.


## 5.4 Validasi HTML Explanation

In [ ]:
# =====================================================
# CELL 34 : VALIDASI HTML EXPLANATION
# =====================================================

print_header("VALIDASI HTML EXPLANATION")

total_html = count_files(

    LOCAL_HTML_DIR,

    ".html"

)

print()

print_info(

    f"Jumlah HTML : {total_html}"

)

print()

if total_html == len(LIME_RESULTS):

    print_success(

        "Seluruh HTML berhasil disimpan."

    )

else:

    print_warning(

        "Masih terdapat HTML yang belum dibuat."

    )


VALIDASI HTML EXPLANATION

ℹ️ Jumlah HTML : 30

✅ Seluruh HTML berhasil disimpan.


# Kode Baru

## 5.6 Persiapan Playwright Renderer

In [ ]:
# =====================================================
# CELL 34A : INSTALL PLAYWRIGHT
# =====================================================

print_header("INSTALL PLAYWRIGHT")

!pip -q install playwright pillow

!playwright install chromium

print()

print_success(

    "Playwright berhasil diinstall."

)


INSTALL PLAYWRIGHT
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 47.4/47.4 MB 23.7 MB/s eta 0:00:00
177 MiB [] 0% 364.7s177 MiB [] 0% 130.5s177 MiB [] 0% 400.8s177 MiB [] 0% 460.2s177 MiB [] 0% 369.1s177 MiB [] 0% 319.7s177 MiB [] 0% 288.3s177 MiB [] 0% 266.5s177 MiB [] 0% 249.0s177 MiB [] 0% 233.7s177 MiB [] 0% 219.1s177 MiB [] 0% 206.9s177 MiB [] 0% 181.8s177 MiB [] 0% 163.7s177 MiB [] 0% 150.0s177 MiB [] 0% 138.9s177 MiB [] 0% 130.3s177 MiB [] 0% 120.0s177 MiB [] 0% 109.8s177 MiB [] 0% 99.7s177 MiB [] 0% 89.9s177 MiB [] 0% 81.9s177 MiB [] 0% 74.4s177 MiB [] 0% 68.5s177 MiB [] 0% 62.0s177 MiB [] 1% 56.9s177 MiB [] 1% 52.2s177 MiB [] 1% 43.9s177 MiB [] 1% 39.6s177 MiB [] 1% 35.8s177 MiB [] 2% 32.3s177 MiB [] 2% 29.3s177 MiB [] 2% 26.4s177 MiB [] 3% 24.5s177 MiB [] 3% 23.0s177 MiB [] 3% 21.8s177 MiB [] 3% 20.4s177 MiB [] 4% 18.0s177 MiB [] 4% 18.2s177 MiB [] 4% 16.8s177 MiB [] 5% 15.3s177 MiB [] 5% 14.4s177 MiB [] 6% 13.4s177 MiB [] 7% 11.9s177 MiB [] 7% 11.3s177 MiB [] 8% 10.8s177 MiB 

## 5.7 Import Playwright

In [ ]:
# =====================================================
# CELL 34B : IMPORT PLAYWRIGHT
# =====================================================

print_header("IMPORT PLAYWRIGHT")

import asyncio
import time
from pathlib import Path

from PIL import Image

from playwright.async_api import async_playwright

print()

print_success(

    "Playwright berhasil diimport."

)


IMPORT PLAYWRIGHT

✅ Playwright berhasil diimport.


## 5.8 Menjalankan Local HTTP Server

In [ ]:
# =====================================================
# CELL 34C : START LOCAL HTTP SERVER
# =====================================================

print_header("START LOCAL HTTP SERVER")

import socket
import threading
from http.server import ThreadingHTTPServer, SimpleHTTPRequestHandler

SERVER_PORT = 8000

ROOT_SERVER = Path(PATHS["XD"]).resolve()

class SilentHTTPRequestHandler(SimpleHTTPRequestHandler):

    def log_message(self, format, *args):
        pass


def start_http_server():

    os.chdir(ROOT_SERVER)

    httpd = ThreadingHTTPServer(

        ("127.0.0.1", SERVER_PORT),

        SilentHTTPRequestHandler

    )

    httpd.serve_forever()


def port_available(port):

    with socket.socket(socket.AF_INET, socket.SOCK_STREAM) as s:

        return s.connect_ex(("127.0.0.1", port)) != 0


if port_available(SERVER_PORT):

    server_thread = threading.Thread(

        target=start_http_server,

        daemon=True

    )

    server_thread.start()

    time.sleep(2)

    print()

    print_info(

        f"HTTP Server : http://127.0.0.1:{SERVER_PORT}"

    )

else:

    print()

    print_info(

        "HTTP Server sudah berjalan."

    )

print()

print_success(

    "Local HTTP Server berhasil dijalankan."

)


START LOCAL HTTP SERVER

ℹ️ HTTP Server : http://127.0.0.1:8000

✅ Local HTTP Server berhasil dijalankan.


## 5.5 Generate LIME Visualization

In [ ]:
# =====================================================
# CELL 35 : GENERATE LIME VISUALIZATION
# =====================================================

print_header("GENERATE LIME VISUALIZATION")

visualization_success = 0

for sample_id, sample_data in tqdm(

    LIME_RESULTS.items(),

    desc="Generate PNG"

):

    explanation = sample_data["lime_explanation"]

    predicted_label = sample_data["predicted_label"]

    figure = explanation.as_pyplot_figure(

        label=predicted_label

    )

    figure.set_size_inches(

        10,

        6

    )

    png_path = os.path.join(

        LOCAL_PNG_DIR,

        f"{sample_id}.png"

    )

    figure.savefig(

        png_path,

        dpi=300,

        bbox_inches="tight"

    )

    plt.close(

        figure

    )

    sample_data["png_path"] = png_path

    visualization_success += 1

print()

print_info(

    f"PNG berhasil : {visualization_success}"

)

print()

print_success(

    "Seluruh visualisasi LIME berhasil dibuat."

)


GENERATE LIME VISUALIZATION


Generate PNG:   0%|          | 0/30 [00:00<?, ?it/s]


ℹ️ PNG berhasil : 30

✅ Seluruh visualisasi LIME berhasil dibuat.


## 5.6 Validasi LIME Visualization

In [ ]:
# =====================================================
# CELL 36 : VALIDASI LIME VISUALIZATION
# =====================================================

print_header("VALIDASI LIME VISUALIZATION")

total_png = count_files(

    LOCAL_PNG_DIR,

    ".png"

)

print()

print_info(

    f"Jumlah PNG : {total_png}"

)

print()

if total_png == len(LIME_RESULTS):

    print_success(

        "Seluruh visualisasi PNG berhasil disimpan."

    )

else:

    print_warning(

        "Masih terdapat PNG yang belum dibuat."

    )


VALIDASI LIME VISUALIZATION

ℹ️ Jumlah PNG : 30

✅ Seluruh visualisasi PNG berhasil disimpan.


## 5.7 Generate Feature Importance CSV

In [ ]:
# =====================================================
# CELL 37 : GENERATE FEATURE CSV
# =====================================================

print_header("GENERATE FEATURE CSV")

csv_success = 0

for sample_id, sample_data in tqdm(

    LIME_RESULTS.items(),

    desc="Generate CSV"

):

    csv_path = os.path.join(

        LOCAL_CSV_DIR,

        f"{sample_id}.csv"

    )

    sample_data["feature_statistics"].to_csv(

        csv_path,

        index=False,

        encoding="utf-8-sig"

    )

    sample_data["csv_path"] = csv_path

    csv_success += 1

print()

print_info(

    f"CSV berhasil : {csv_success}"

)

print()

print_success(

    "Seluruh CSV berhasil dibuat."

)


GENERATE FEATURE CSV


Generate CSV:   0%|          | 0/30 [00:00<?, ?it/s]


ℹ️ CSV berhasil : 30

✅ Seluruh CSV berhasil dibuat.


## 5.8 Validasi Feature CSV

In [ ]:
# =====================================================
# CELL 38 : VALIDASI FEATURE CSV
# =====================================================

print_header("VALIDASI FEATURE CSV")

total_csv = count_files(

    LOCAL_CSV_DIR,

    ".csv"

)

print()

print_info(

    f"Jumlah CSV : {total_csv}"

)

print()

if total_csv == len(LIME_RESULTS):

    print_success(

        "Seluruh Feature CSV berhasil disimpan."

    )

else:

        print_warning(

            "Masih terdapat CSV yang belum dibuat."

        )


VALIDASI FEATURE CSV

ℹ️ Jumlah CSV : 30

✅ Seluruh Feature CSV berhasil disimpan.


## 5.9 Generate Metadata JSON

In [ ]:
# =====================================================
# CELL 39 : GENERATE METADATA JSON
# =====================================================

print_header("GENERATE METADATA JSON")

metadata_success = 0

for sample_id, sample_data in tqdm(

    LIME_RESULTS.items(),

    desc="Generate Metadata JSON"

):

    metadata_path = os.path.join(

        LOCAL_METADATA_DIR,

        f"{sample_id}.json"

    )

    with open(

        metadata_path,

        "w",

        encoding="utf-8"

    ) as f:

        json.dump(

            sample_data["metadata"],

            f,

            indent=4,

            ensure_ascii=False

        )

    sample_data["metadata_path"] = metadata_path

    metadata_success += 1

print()

print_info(

    f"Metadata JSON : {metadata_success}"

)

print()

print_success(

    "Metadata JSON berhasil dibuat."

)


GENERATE METADATA JSON


Generate Metadata JSON:   0%|          | 0/30 [00:00<?, ?it/s]


ℹ️ Metadata JSON : 30

✅ Metadata JSON berhasil dibuat.


## 5.10 Validasi Metadata JSON

In [ ]:
# =====================================================
# CELL 40 : VALIDASI METADATA JSON
# =====================================================

print_header("VALIDASI METADATA JSON")

total_json = count_files(

    LOCAL_METADATA_DIR,

    ".json"

)

print()

print_info(

    f"Jumlah JSON : {total_json}"

)

print()

if total_json == len(LIME_RESULTS):

    print_success(

        "Seluruh Metadata JSON berhasil disimpan."

    )

else:

    print_warning(

        "Masih terdapat Metadata JSON yang belum dibuat."

    )


VALIDASI METADATA JSON

ℹ️ Jumlah JSON : 30

✅ Seluruh Metadata JSON berhasil disimpan.


## 5.11 Validasi Akhir LIME Local Explanation

In [ ]:
# =====================================================
# CELL 41 : VALIDASI AKHIR LIME LOCAL
# =====================================================

print_header("VALIDASI AKHIR LIME LOCAL")

validation = {

    "HTML":

        count_files(

            LOCAL_HTML_DIR,

            ".html"

        ),

    "PNG":

        count_files(

            LOCAL_PNG_DIR,

            ".png"

        ),

    "CSV":

        count_files(

            LOCAL_CSV_DIR,

            ".csv"

        ),

    "Metadata":

        count_files(

            LOCAL_METADATA_DIR,

            ".json"

        )

}

validation_df = pd.DataFrame({

    "Komponen":

        validation.keys(),

    "Jumlah":

        validation.values()

})

display(

    validation_df

)

print()

expected = len(LIME_RESULTS)

success = True

for value in validation.values():

    if value != expected:

        success = False

print_info(

    f"Jumlah Sampel : {expected}"

)

print()

if success:

    print_success(

        "Seluruh output LIME Local berhasil dibuat."

    )

else:

    print_warning(

        "Masih terdapat output yang belum lengkap."

    )


VALIDASI AKHIR LIME LOCAL


,Komponen,Jumlah
0,HTML,30
1,PNG,30
2,CSV,30
3,Metadata,30



ℹ️ Jumlah Sampel : 30

✅ Seluruh output LIME Local berhasil dibuat.


# 6. GLOBAL LIME ANALYSIS

## 6.1 Persiapan Global Feature Statistics

In [ ]:
# =====================================================
# CELL 42 : PERSIAPAN GLOBAL FEATURE STATISTICS
# =====================================================

print_header("PERSIAPAN GLOBAL FEATURE STATISTICS")

global_feature_statistics = []

processed_sample_ids = []

for sample_id, sample_data in LIME_RESULTS.items():

    global_feature_statistics.append(

        sample_data["feature_statistics"]

    )

    processed_sample_ids.append(

        sample_id

    )

print()

print_info(

    f"Jumlah Feature Statistics : {len(global_feature_statistics)}"

)

print_info(

    f"Jumlah Sampel : {len(processed_sample_ids)}"

)

print()

print_success(

    "Global Feature Statistics berhasil dipersiapkan."

)


PERSIAPAN GLOBAL FEATURE STATISTICS

ℹ️ Jumlah Feature Statistics : 30
ℹ️ Jumlah Sampel : 30

✅ Global Feature Statistics berhasil dipersiapkan.


## 6.2 Validasi Global Feature Statistics

In [ ]:
# =====================================================
# CELL 43 : VALIDASI GLOBAL FEATURE STATISTICS
# =====================================================

print_header("VALIDASI GLOBAL FEATURE STATISTICS")

expected = len(LIME_RESULTS)

obtained = len(global_feature_statistics)

print()

print_info(

    f"Expected : {expected}"

)

print_info(

    f"Obtained : {obtained}"

)

print()

if expected == obtained:

    print_success(

        "Seluruh Feature Statistics berhasil dikumpulkan."

    )

else:

    print_warning(

        "Jumlah Feature Statistics tidak sesuai."

    )


VALIDASI GLOBAL FEATURE STATISTICS

ℹ️ Expected : 30
ℹ️ Obtained : 30

✅ Seluruh Feature Statistics berhasil dikumpulkan.


## 6.3 Generate Global Token Importance

In [ ]:
# =====================================================
# CELL 44 : GENERATE GLOBAL TOKEN IMPORTANCE
# =====================================================

print_header("GENERATE GLOBAL TOKEN IMPORTANCE")

from collections import defaultdict

token_statistics = defaultdict(

    lambda:{

        "sum_abs_weight":0.0,

        "sum_weight":0.0,

        "frequency":0

    }

)

for feature_df in global_feature_statistics:

    for _, row in feature_df.iterrows():

        token = str(

            row["Token"]

        ).strip()

        if token == "":

            continue

        token_statistics[token]["sum_abs_weight"] += float(

            row["|Weight|"]

        )

        token_statistics[token]["sum_weight"] += float(

            row["Weight"]

        )

        token_statistics[token]["frequency"] += 1

global_token_df = (

    pd.DataFrame(

        token_statistics

    )

    .T

    .reset_index()

    .rename(

        columns={

            "index":"Token"

        }

    )

)

global_token_df["mean_abs_weight"] = (

    global_token_df["sum_abs_weight"]

    /

    global_token_df["frequency"]

)

global_token_df["mean_weight"] = (

    global_token_df["sum_weight"]

    /

    global_token_df["frequency"]

)

global_token_df = global_token_df.sort_values(

    by="mean_abs_weight",

    ascending=False

).reset_index(

    drop=True

)

display(

    global_token_df.head(20)

)

print()

print_success(

    "Global Token Importance berhasil dibuat."

)


GENERATE GLOBAL TOKEN IMPORTANCE


,Token,sum_abs_weight,sum_weight,frequency,mean_abs_weight,mean_weight
0,parah,0.953833,0.953833,1.0,0.953833,0.953833
1,maaf,0.572378,0.572378,1.0,0.572378,0.572378
2,bagus,0.412554,0.412554,1.0,0.412554,0.412554
3,100,0.378919,-0.378919,1.0,0.378919,-0.378919
4,kenapa,0.753019,0.753019,2.0,0.376509,0.376509
5,oknum,0.341514,0.341514,1.0,0.341514,0.341514
6,guna,0.313976,-0.313976,1.0,0.313976,-0.313976
7,masih,0.309839,0.309839,1.0,0.309839,0.309839
8,tak,0.293524,0.293524,1.0,0.293524,0.293524
9,susah,0.278015,0.278015,1.0,0.278015,0.278015



✅ Global Token Importance berhasil dibuat.


## 6.4 Validasi Global Token Importance

In [ ]:
# =====================================================
# CELL 45 : VALIDASI GLOBAL TOKEN IMPORTANCE
# =====================================================

print_header("VALIDASI GLOBAL TOKEN IMPORTANCE")

print()

print_info(

    f"Jumlah Token : {len(global_token_df)}"

)

print_info(

    f"Top Token : {global_token_df.iloc[0]['Token']}"

)

print_info(

    f"Mean |Weight| : {global_token_df.iloc[0]['mean_abs_weight']:.6f}"

)

print()

print_success(

    "Global Token Importance berhasil divalidasi."

)


VALIDASI GLOBAL TOKEN IMPORTANCE

ℹ️ Jumlah Token : 184
ℹ️ Top Token : parah
ℹ️ Mean |Weight| : 0.953833

✅ Global Token Importance berhasil divalidasi.


## 6.5 Generate Global Bar Plot

In [ ]:
# =====================================================
# CELL 46 : GENERATE GLOBAL BAR PLOT
# =====================================================

print_header("GENERATE GLOBAL BAR PLOT")

TOP_N = 20

plot_df = global_token_df.head(

    TOP_N

).copy()

plt.figure(

    figsize=(12,8)

)

plt.barh(

    plot_df["Token"],

    plot_df["mean_abs_weight"]

)

plt.gca().invert_yaxis()

plt.xlabel(

    "Mean Absolute Weight"

)

plt.ylabel(

    "Token"

)

plt.title(

    "Global LIME Feature Importance"

)

global_bar_path = os.path.join(

    GLOBAL_DIR,

    "global_lime_feature_importance.png"

)

plt.savefig(

    global_bar_path,

    dpi=300,

    bbox_inches="tight"

)

plt.close()

print()

print_info(

    f"Output : {global_bar_path}"

)

print()

print_success(

    "Global Bar Plot berhasil dibuat."

)


GENERATE GLOBAL BAR PLOT

ℹ️ Output : /content/drive/MyDrive/Skripsi_IndoBERT/xai/XD_LIME_Analysis/global/global_lime_feature_importance.png

✅ Global Bar Plot berhasil dibuat.


## 6.6 Validasi Global Bar Plot

In [ ]:
# =====================================================
# CELL 47 : VALIDASI GLOBAL BAR PLOT
# =====================================================

print_header("VALIDASI GLOBAL BAR PLOT")

print()

print_info(

    f"File : {os.path.basename(global_bar_path)}"

)

print_info(

    f"Lokasi : {global_bar_path}"

)

print()

if os.path.exists(

    global_bar_path

):

    print_success(

        "Global Bar Plot berhasil disimpan."

    )

else:

    raise FileNotFoundError(

        global_bar_path

    )


VALIDASI GLOBAL BAR PLOT

ℹ️ File : global_lime_feature_importance.png
ℹ️ Lokasi : /content/drive/MyDrive/Skripsi_IndoBERT/xai/XD_LIME_Analysis/global/global_lime_feature_importance.png

✅ Global Bar Plot berhasil disimpan.


## 6.7 Generate Overall LIME Analysis

In [ ]:
# =====================================================
# CELL 48 : GENERATE OVERALL LIME ANALYSIS
# =====================================================

print_header("GENERATE OVERALL LIME ANALYSIS")

TOP_N = 20

overall_summary = global_token_df.head(

    TOP_N

).copy()

overall_summary["Ranking"] = np.arange(

    1,

    len(overall_summary)+1

)

overall_summary = overall_summary[[

    "Ranking",

    "Token",

    "frequency",

    "mean_abs_weight",

    "mean_weight"

]]

display(

    overall_summary

)

print()

print_success(

    "Overall LIME Analysis berhasil dibuat."

)


GENERATE OVERALL LIME ANALYSIS


,Ranking,Token,frequency,mean_abs_weight,mean_weight
0,1,parah,1.0,0.953833,0.953833
1,2,maaf,1.0,0.572378,0.572378
2,3,bagus,1.0,0.412554,0.412554
3,4,100,1.0,0.378919,-0.378919
4,5,kenapa,2.0,0.376509,0.376509
5,6,oknum,1.0,0.341514,0.341514
6,7,guna,1.0,0.313976,-0.313976
7,8,masih,1.0,0.309839,0.309839
8,9,tak,1.0,0.293524,0.293524
9,10,susah,1.0,0.278015,0.278015



✅ Overall LIME Analysis berhasil dibuat.


## 6.8 Validasi Overall LIME Analysis

In [ ]:
# =====================================================
# CELL 49 : VALIDASI OVERALL LIME ANALYSIS
# =====================================================

print_header("VALIDASI OVERALL LIME ANALYSIS")

print()

print_info(

    f"Jumlah Token : {len(overall_summary)}"

)

print_info(

    f"Token Ranking 1 : {overall_summary.iloc[0]['Token']}"

)

print_info(

    f"Mean |Weight| : {overall_summary.iloc[0]['mean_abs_weight']:.6f}"

)

print()

print_success(

    "Overall LIME Analysis berhasil divalidasi."

)


VALIDASI OVERALL LIME ANALYSIS

ℹ️ Jumlah Token : 20
ℹ️ Token Ranking 1 : parah
ℹ️ Mean |Weight| : 0.953833

✅ Overall LIME Analysis berhasil divalidasi.


# 7. PENYIMPANAN HASIL GLOBAL LIME

## 7.1 Simpan Overall LIME Analysis

In [ ]:
# =====================================================
# CELL 50 : SIMPAN OVERALL LIME ANALYSIS
# =====================================================

print_header("SIMPAN OVERALL LIME ANALYSIS")

overall_csv_path = os.path.join(

    CSV_DIR,

    "overall_lime_analysis.csv"

)

overall_summary.to_csv(

    overall_csv_path,

    index=False,

    encoding="utf-8-sig"

)

print()

print_info(

    f"Output : {overall_csv_path}"

)

print()

print_success(

    "Overall LIME Analysis berhasil disimpan."

)


SIMPAN OVERALL LIME ANALYSIS

ℹ️ Output : /content/drive/MyDrive/Skripsi_IndoBERT/xai/XD_LIME_Analysis/csv/overall_lime_analysis.csv

✅ Overall LIME Analysis berhasil disimpan.


## 7.2 Validasi Overall LIME Analysis

In [ ]:
# =====================================================
# CELL 51 : VALIDASI OVERALL LIME ANALYSIS CSV
# =====================================================

print_header("VALIDASI OVERALL LIME ANALYSIS CSV")

df = pd.read_csv(

    overall_csv_path

)

print()

print_info(

    f"Jumlah Baris : {len(df)}"

)

print_info(

    f"Jumlah Kolom : {len(df.columns)}"

)

print()

if os.path.exists(

    overall_csv_path

):

    print_success(

        "Overall CSV berhasil disimpan."

    )

else:

    raise FileNotFoundError(

        overall_csv_path

    )


VALIDASI OVERALL LIME ANALYSIS CSV

ℹ️ Jumlah Baris : 20
ℹ️ Jumlah Kolom : 5

✅ Overall CSV berhasil disimpan.


## 7.3 Simpan Global Token Importance

In [ ]:
# =====================================================
# CELL 52 : SIMPAN GLOBAL TOKEN IMPORTANCE
# =====================================================

print_header("SIMPAN GLOBAL TOKEN IMPORTANCE")

global_csv_path = os.path.join(

    CSV_DIR,

    "global_token_importance.csv"

)

global_token_df.to_csv(

    global_csv_path,

    index=False,

    encoding="utf-8-sig"

)

print()

print_info(

    f"Output : {global_csv_path}"

)

print()

print_success(

    "Global Token Importance berhasil disimpan."

)


SIMPAN GLOBAL TOKEN IMPORTANCE

ℹ️ Output : /content/drive/MyDrive/Skripsi_IndoBERT/xai/XD_LIME_Analysis/csv/global_token_importance.csv

✅ Global Token Importance berhasil disimpan.


## 7.4 Validasi Global Token Importance

In [ ]:
# =====================================================
# CELL 53 : VALIDASI GLOBAL TOKEN IMPORTANCE CSV
# =====================================================

print_header("VALIDASI GLOBAL TOKEN IMPORTANCE CSV")

df = pd.read_csv(

    global_csv_path

)

print()

print_info(

    f"Jumlah Token : {len(df)}"

)

print()

if os.path.exists(

    global_csv_path

):

    print_success(

        "Global Token CSV berhasil disimpan."

    )

else:

    raise FileNotFoundError(

        global_csv_path

    )


VALIDASI GLOBAL TOKEN IMPORTANCE CSV

ℹ️ Jumlah Token : 184

✅ Global Token CSV berhasil disimpan.


## 7.5 Validasi Akhir Global LIME

In [ ]:
# =====================================================
# CELL 54 : VALIDASI AKHIR GLOBAL LIME
# =====================================================

print_header("VALIDASI AKHIR GLOBAL LIME")

validation = {

    "Global Plot":

        count_files(

            GLOBAL_DIR,

            ".png"

        ),

    "Global CSV":

        count_files(

            CSV_DIR,

            ".csv"

        )

}

validation_df = pd.DataFrame({

    "Komponen":

        validation.keys(),

    "Jumlah":

        validation.values()

})

display(

    validation_df

)

print()

print_success(

    "Seluruh output Global LIME berhasil dibuat."

)


VALIDASI AKHIR GLOBAL LIME


,Komponen,Jumlah
0,Global Plot,1
1,Global CSV,2



✅ Seluruh output Global LIME berhasil dibuat.


# 8. RINGKASAN HASIL NOTEBOOK XD

## 8.1 Ringkasan Output Notebook

In [ ]:
# =====================================================
# CELL 55 : RINGKASAN OUTPUT NOTEBOOK
# =====================================================

print_header("RINGKASAN OUTPUT NOTEBOOK")

summary = pd.DataFrame({

    "Komponen":[

        "LIME Explanation",

        "HTML",

        "PNG",

        "CSV",

        "Metadata JSON",

        "Global Plot",

        "Global CSV"

    ],

    "Jumlah":[

        len(LIME_RESULTS),

        count_files(

            LOCAL_HTML_DIR,

            ".html"

        ),

        count_files(

            LOCAL_PNG_DIR,

            ".png"

        ),

        count_files(

            LOCAL_CSV_DIR,

            ".csv"

        ),

        count_files(

            LOCAL_METADATA_DIR,

            ".json"

        ),

        count_files(

            GLOBAL_DIR,

            ".png"

        ),

        count_files(

            CSV_DIR,

            ".csv"

        )

    ]

})

display(summary)

print()

print_success(

    "Ringkasan output notebook berhasil dibuat."

)


RINGKASAN OUTPUT NOTEBOOK


,Komponen,Jumlah
0,LIME Explanation,30
1,HTML,30
2,PNG,30
3,CSV,30
4,Metadata JSON,30
5,Global Plot,1
6,Global CSV,2



✅ Ringkasan output notebook berhasil dibuat.


## 8.2 Validasi Ringkasan Output

In [ ]:
# =====================================================
# CELL 56 : VALIDASI RINGKASAN OUTPUT
# =====================================================

print_header("VALIDASI RINGKASAN OUTPUT")

expected = len(LIME_RESULTS)

success = True

checks = {

    "HTML":

        count_files(

            LOCAL_HTML_DIR,

            ".html"

        ),

    "PNG":

        count_files(

            LOCAL_PNG_DIR,

            ".png"

        ),

    "CSV":

        count_files(

            LOCAL_CSV_DIR,

            ".csv"

        ),

    "JSON":

        count_files(

            LOCAL_METADATA_DIR,

            ".json"

        )

}

for value in checks.values():

    if value != expected:

        success = False

print()

print_info(

    f"Jumlah Sampel : {expected}"

)

print()

if success:

    print_success(

        "Seluruh output notebook lengkap."

    )

else:

    print_warning(

        "Masih terdapat output yang belum lengkap."

    )


VALIDASI RINGKASAN OUTPUT

ℹ️ Jumlah Sampel : 30

✅ Seluruh output notebook lengkap.


# 9. PENUTUP NOTEBOOK XD

## 9.1 Generate Manifest

In [ ]:
# =====================================================
# CELL 57 : GENERATE MANIFEST
# =====================================================

print_header("GENERATE MANIFEST")

manifest = {

    "project":"XD_LIME_Analysis",

    "total_sample":len(LIME_RESULTS),

    "total_token":len(global_token_df),

    "total_overall_token":len(overall_summary),

    "html":

        count_files(

            LOCAL_HTML_DIR,

            ".html"

        ),

    "png":

        count_files(

            LOCAL_PNG_DIR,

            ".png"

        ),

    "csv":

        count_files(

            LOCAL_CSV_DIR,

            ".csv"

        ),

    "metadata":

        count_files(

            LOCAL_METADATA_DIR,

            ".json"

        ),

    "global_plot":

        count_files(

            GLOBAL_DIR,

            ".png"

        ),

    "global_csv":

        count_files(

            CSV_DIR,

            ".csv"

        )

}

manifest_path = os.path.join(

    ROOT_OUTPUT_DIR,

    "manifest_xd.json"

)

with open(

    manifest_path,

    "w",

    encoding="utf-8"

) as f:

    json.dump(

        manifest,

        f,

        indent=4,

        ensure_ascii=False

    )

print()

print_info(

    f"Manifest : {manifest_path}"

)

print()

print_success(

    "Manifest berhasil dibuat."

)


GENERATE MANIFEST

ℹ️ Manifest : /content/drive/MyDrive/Skripsi_IndoBERT/xai/XD_LIME_Analysis/manifest_xd.json

✅ Manifest berhasil dibuat.


## 9.2 Notebook XD Completed

In [ ]:
# =====================================================
# CELL 57 : GENERATE MANIFEST
# =====================================================

print_header("GENERATE MANIFEST")

manifest = {

    "project":"XD_LIME_Analysis",

    "total_sample":len(LIME_RESULTS),

    "total_token":len(global_token_df),

    "total_overall_token":len(overall_summary),

    "html":

        count_files(

            LOCAL_HTML_DIR,

            ".html"

        ),

    "png":

        count_files(

            LOCAL_PNG_DIR,

            ".png"

        ),

    "csv":

        count_files(

            LOCAL_CSV_DIR,

            ".csv"

        ),

    "metadata":

        count_files(

            LOCAL_METADATA_DIR,

            ".json"

        ),

    "global_plot":

        count_files(

            GLOBAL_DIR,

            ".png"

        ),

    "global_csv":

        count_files(

            CSV_DIR,

            ".csv"

        )

}

manifest_path = os.path.join(

    ROOT_OUTPUT_DIR,

    "manifest_xd.json"

)

with open(

    manifest_path,

    "w",

    encoding="utf-8"

) as f:

    json.dump(

        manifest,

        f,

        indent=4,

        ensure_ascii=False

    )

print()

print_info(

    f"Manifest : {manifest_path}"

)

print()

print_success(

    "Manifest berhasil dibuat."

)


GENERATE MANIFEST

ℹ️ Manifest : /content/drive/MyDrive/Skripsi_IndoBERT/xai/XD_LIME_Analysis/manifest_xd.json

✅ Manifest berhasil dibuat.


tambahan untuk ppt

In [ ]:
print_header("LOAD SAMPLE UNTUK PERHITUNGAN MANUAL")

MANUAL_SAMPLE_ID = "XAI_002"

sample_data = LIME_RESULTS[MANUAL_SAMPLE_ID]

text = sample_data["text"]

predicted_label = sample_data["predicted_label"]

predicted_sentiment = sample_data["predicted_sentiment"]

probabilities = sample_data["probabilities"]

print()

print_info(f"Sample ID : {MANUAL_SAMPLE_ID}")

print_info(f"Prediksi : {predicted_sentiment}")

print_info(f"Confidence : {probabilities[predicted_label]:.6f}")

print()

print("Teks Ulasan")

print(text)

print()

print_success(
    "Sample berhasil dimuat."
)


LOAD SAMPLE UNTUK PERHITUNGAN MANUAL

ℹ️ Sample ID : XAI_002
ℹ️ Prediksi : Positif
ℹ️ Confidence : 0.998164

Teks Ulasan
mudah dan sangat membantu sekali

✅ Sample berhasil dimuat.


In [ ]:
print_header("TOKENISASI DAN MENENTUKAN TOKEN TARGET")

# ==========================================================
# Tokenisasi sederhana (sesuai cara kerja awal LIME)
# ==========================================================

tokens = text.split()

print()

print_info(f"Jumlah Token : {len(tokens)}")

print()

print("Daftar Token")

for i, token in enumerate(tokens):
    print(f"{i+1}. {token}")

print()

# ==========================================================
# Tentukan token target
# ==========================================================

TARGET_TOKEN = "membantu"

if TARGET_TOKEN not in tokens:
    raise ValueError(f"Token '{TARGET_TOKEN}' tidak ditemukan.")

target_index = tokens.index(TARGET_TOKEN)

other_tokens = tokens.copy()
other_tokens.remove(TARGET_TOKEN)

print_info(f"Target Token : {TARGET_TOKEN}")

print()

print("Token lain")

for token in other_tokens:
    print("-", token)

print()

print_success(
    "Token target berhasil ditentukan."
)


TOKENISASI DAN MENENTUKAN TOKEN TARGET

ℹ️ Jumlah Token : 5

Daftar Token
1. mudah
2. dan
3. sangat
4. membantu
5. sekali

ℹ️ Target Token : membantu

Token lain
- mudah
- dan
- sangat
- sekali

✅ Token target berhasil ditentukan.


In [ ]:
#54. membangun representasi fitur lime

print_header("MEMBANGUN REPRESENTASI FITUR LIME")

# ==========================================================
# Gunakan sample XAI_010
# ==========================================================

MANUAL_SAMPLE_ID = "XAI_010"

sample_data = LIME_RESULTS[MANUAL_SAMPLE_ID]

text = sample_data["text"]

predicted_label = sample_data["predicted_label"]

probabilities = sample_data["probabilities"]

# ==========================================================
# Tokenisasi
# ==========================================================

tokens = text.split()

print()

print_info(f"Sample : {MANUAL_SAMPLE_ID}")

print_info(f"Teks : {text}")

print()

print("Daftar Token")

for i, token in enumerate(tokens):
    print(f"{i} -> {token}")

print()

# ==========================================================
# Target Token
# ==========================================================

TARGET_TOKEN = "gak"

if TARGET_TOKEN not in tokens:
    raise ValueError(
        f"Token '{TARGET_TOKEN}' tidak ditemukan."
    )

target_index = tokens.index(TARGET_TOKEN)

# ==========================================================
# Token selain target
# ==========================================================

feature_tokens = [
    token
    for token in tokens
    if token != TARGET_TOKEN
]

print_info(f"Target Token : {TARGET_TOKEN}")

print()

print("Feature Tokens")

for token in feature_tokens:
    print("-", token)

print()

# ==========================================================
# Representasi biner fitur
# ==========================================================

feature_map = {

    i: token

    for i, token in enumerate(feature_tokens)

}

display(

    pd.DataFrame({

        "Feature": list(feature_map.keys()),

        "Token": list(feature_map.values())

    })

)

print()

print_success(
    "Representasi fitur berhasil dibuat."
)


MEMBANGUN REPRESENTASI FITUR LIME

ℹ️ Sample : XAI_010
ℹ️ Teks : aplikasi gak guna

Daftar Token
0 -> aplikasi
1 -> gak
2 -> guna

ℹ️ Target Token : gak

Feature Tokens
- aplikasi
- guna



,Feature,Token
0,0,aplikasi
1,1,guna



✅ Representasi fitur berhasil dibuat.


In [ ]:
#55. membangkitkan seuruh perturbation

print_header("MEMBANGKITKAN SELURUH PERTURBATION")

from itertools import product

# ==========================================================
# Semua kombinasi feature token
# ==========================================================

binary_vectors = list(

    product(

        [0, 1],

        repeat=len(feature_tokens)

    )

)

perturbation_df = pd.DataFrame(

    binary_vectors,

    columns=feature_tokens

)

# ==========================================================
# Bentuk Text S
# ==========================================================

texts_S = []

texts_S_plus_i = []

coalitions = []

for row in binary_vectors:

    coalition = []

    for keep, token in zip(row, feature_tokens):

        if keep == 1:

            coalition.append(token)

    coalitions.append(coalition)

    text_S = " ".join(coalition)

    texts_S.append(text_S)

    text_Si = coalition.copy()

    text_Si.append(TARGET_TOKEN)

    text_S_plus_i = " ".join(text_Si)

    texts_S_plus_i.append(text_S_plus_i)

perturbation_df["Coalition"] = coalitions

perturbation_df["Text_S"] = texts_S

perturbation_df["Text_S_plus_i"] = texts_S_plus_i

# ==========================================================
# Rapikan urutan kolom
# ==========================================================

perturbation_df = perturbation_df[

    [

        "Coalition",

        "Text_S",

        "Text_S_plus_i",

        *feature_tokens

    ]

]

print()

display(perturbation_df)

print()

print_info(

    f"Jumlah Perturbation : {len(perturbation_df)}"

)

print()

print_success(

    "Perturbation berhasil dibuat."

)


MEMBANGKITKAN SELURUH PERTURBATION



,Coalition,Text_S,Text_S_plus_i,aplikasi,guna
0,[],,gak,0,0
1,[guna],guna,guna gak,0,1
2,[aplikasi],aplikasi,aplikasi gak,1,0
3,"[aplikasi, guna]",aplikasi guna,aplikasi guna gak,1,1



ℹ️ Jumlah Perturbation : 4

✅ Perturbation berhasil dibuat.


In [ ]:
print_header("PREDIKSI MODEL UNTUK SETIAP PERTURBATION")

# ==========================================================
# 56. Hitung probabilitas seluruh perturbation
# ==========================================================

probabilities = predict_proba(

    perturbation_df["Text_S_plus_i"].tolist()

)

# ==========================================================
# Simpan probabilitas
# ==========================================================

perturbation_df["Prob_Negatif"] = probabilities[:,0]

perturbation_df["Prob_Netral"] = probabilities[:,1]

perturbation_df["Prob_Positif"] = probabilities[:,2]

# ==========================================================
# Prediksi kelas
# ==========================================================

predicted_class = np.argmax(

    probabilities,

    axis=1

)

perturbation_df["Prediksi"] = [

    id2label[x]

    for x in predicted_class

]

print()

display(

    perturbation_df[
        [

            "Text_S",

            "Text_S_plus_i",

            "Prob_Negatif",

            "Prob_Netral",

            "Prob_Positif",

            "Prediksi"

        ]
    ]

)

print()

print_success(

    "Probabilitas seluruh perturbation berhasil dihitung."

)


PREDIKSI MODEL UNTUK SETIAP PERTURBATION



,Text_S,Text_S_plus_i,Prob_Negatif,Prob_Netral,Prob_Positif,Prediksi
0,,gak,0.950727,0.010897,0.038376,Negatif
1,guna,guna gak,0.954408,0.007900,0.037692,Negatif
2,aplikasi,aplikasi gak,0.994575,0.002930,0.002495,Negatif
3,aplikasi guna,aplikasi guna gak,0.993585,0.002297,0.004117,Negatif



✅ Probabilitas seluruh perturbation berhasil dihitung.


ξ(x)=argg∈Gmin​[L(f,g,πx​)+Ω(g)]

In [ ]:
# 57. Hitung cosine distance

print_header("MENGHITUNG COSINE DISTANCE")

from sklearn.feature_extraction.text import CountVectorizer
from sklearn.metrics.pairwise import cosine_distances

# ==========================================================
# Dokumen asli + seluruh perturbation
# ==========================================================

documents = [text] + perturbation_df["Text_S_plus_i"].tolist()

# ==========================================================
# Representasi Bag-of-Words
# ==========================================================

vectorizer = CountVectorizer()

X = vectorizer.fit_transform(documents)

# ==========================================================
# Cosine Distance
# ==========================================================

distance = cosine_distances(

    X[0],

    X[1:]

).flatten()

perturbation_df["Cosine_Distance"] = distance

print()

display(

    perturbation_df[
        [

            "Text_S_plus_i",

            "Cosine_Distance"

        ]
    ]

)

print()

print_success(
    "Cosine Distance berhasil dihitung."
)


MENGHITUNG COSINE DISTANCE



,Text_S_plus_i,Cosine_Distance
0,gak,0.422650
1,guna gak,0.183503
2,aplikasi gak,0.183503
3,aplikasi guna gak,0.000000



✅ Cosine Distance berhasil dihitung.


Sekarang gunakan rumus kernel LIME

π
x
	​

(z)=exp(−
σ
2
D(x,z)
2
	​

)

In [ ]:
print_header("MENGHITUNG KERNEL WEIGHT")

# ==========================================================
# 58. Kernel Width
# ==========================================================

kernel_width = 0.75

# ==========================================================
# Kernel Similarity
# ==========================================================

weights = np.exp(

    -(

        perturbation_df["Cosine_Distance"]**2

    ) /

    (kernel_width**2)

)

perturbation_df["Kernel_Weight"] = weights

print()

display(

    perturbation_df[
        [

            "Text_S_plus_i",

            "Cosine_Distance",

            "Kernel_Weight"

        ]
    ]

)

print()

print_success(
    "Kernel Weight berhasil dihitung."
)


MENGHITUNG KERNEL WEIGHT



,Text_S_plus_i,Cosine_Distance,Kernel_Weight
0,gak,0.422650,0.727916
1,guna gak,0.183503,0.941893
2,aplikasi gak,0.183503,0.941893
3,aplikasi guna gak,0.000000,1.000000



✅ Kernel Weight berhasil dihitung.


In [ ]:
# =====================================================
# CELL 59 : MEMBENTUK MATRIKS REGRESI LIME (X)
# =====================================================

print_header("MEMBENTUK MATRIKS REGRESI LIME (X)")

# ==========================================================
# LANGKAH 1
# Ambil representasi biner perturbasi
# ==========================================================

X = perturbation_df[
    feature_tokens
].values.astype(float)

# ==========================================================
# LANGKAH 2
# Tambahkan Intercept
# g(z)=β0+β1z1+β2z2
# ==========================================================

X = np.column_stack(

    [

        np.ones(len(X)),

        X

    ]

)

# ==========================================================
# LANGKAH 3
# Nama Kolom
# ==========================================================

column_names = [

    "Intercept",

    *feature_tokens

]

X_df = pd.DataFrame(

    X,

    columns=column_names

)

print()

display(X_df)

print()

print_info(

    f"Shape Matrix X : {X.shape}"

)

print()

print_success(

    "Matrix X berhasil dibuat."

)


MEMBENTUK MATRIKS REGRESI LIME (X)



,Intercept,aplikasi,guna
0,1.0,0.0,0.0
1,1.0,0.0,1.0
2,1.0,1.0,0.0
3,1.0,1.0,1.0



ℹ️ Shape Matrix X : (4, 3)

✅ Matrix X berhasil dibuat.


In [ ]:
# =====================================================
# CELL 60 : MEMBENTUK VEKTOR TARGET (y)
# =====================================================

print_header("MEMBENTUK VEKTOR TARGET (y)")

# ==========================================================
# LANGKAH 1
# Target regresi menggunakan probabilitas
# kelas hasil prediksi model
# ==========================================================

y = perturbation_df[

    "Prob_Negatif"

].values.astype(float)

# ==========================================================
# LANGKAH 2
# Ubah menjadi DataFrame
# ==========================================================

y_df = pd.DataFrame(

    {

        "y=f(z)": y

    }

)

print()

display(y_df)

print()

print_info(

    f"Panjang y : {len(y)}"

)

print()

print_success(

    "Vector target berhasil dibuat."

)


MEMBENTUK VEKTOR TARGET (y)



,y=f(z)
0,0.950727
1,0.954408
2,0.994575
3,0.993585



ℹ️ Panjang y : 4

✅ Vector target berhasil dibuat.


In [ ]:
# =====================================================
# CELL 61 : MEMBENTUK MATRIKS BOBOT (W)
# =====================================================

print_header("MEMBENTUK MATRIKS BOBOT (W)")

# ==========================================================
# LANGKAH 1
# Ambil Kernel Weight
# ==========================================================

weights = perturbation_df[

    "Kernel_Weight"

].values.astype(float)

# ==========================================================
# LANGKAH 2
# Bentuk matriks diagonal
# ==========================================================

W = np.diag(

    weights

)

print()

display(

    pd.DataFrame(W)

)

print()

print_info(

    f"Shape W : {W.shape}"

)

print()

print_success(

    "Matrix W berhasil dibuat."

)


MEMBENTUK MATRIKS BOBOT (W)



,0,1,2,3
0,0.727916,0.000000,0.000000,0.0
1,0.000000,0.941893,0.000000,0.0
2,0.000000,0.000000,0.941893,0.0
3,0.000000,0.000000,0.000000,1.0



ℹ️ Shape W : (4, 4)

✅ Matrix W berhasil dibuat.


β=(XTWX)−1XTWy​

In [ ]:
# =====================================================
# CELL 62 : HITUNG KOEFISIEN REGRESI LIME (β)
# =====================================================

print_header("MENGHITUNG KOEFISIEN REGRESI LIME")

# ==========================================================
# LANGKAH 1
# Hitung Transpose Matrix X
# ==========================================================

XT = X.T

print()
print("="*70)
print("LANGKAH 1 : Xᵀ")
print("="*70)

display(

    pd.DataFrame(

        XT,

        index=column_names

    )

)

# ==========================================================
# LANGKAH 2
# Hitung XᵀW
# ==========================================================

XTW = XT @ W

print()
print("="*70)
print("LANGKAH 2 : XᵀW")
print("="*70)

display(

    pd.DataFrame(

        XTW,

        index=column_names

    )

)

# ==========================================================
# LANGKAH 3
# Hitung XᵀWX
# ==========================================================

XTWX = XTW @ X

print()
print("="*70)
print("LANGKAH 3 : XᵀWX")
print("="*70)

display(

    pd.DataFrame(

        XTWX,

        index=column_names,

        columns=column_names

    )

)

# ==========================================================
# LANGKAH 4
# Hitung Invers
# ==========================================================

XTWX_inv = np.linalg.inv(XTWX)

print()
print("="*70)
print("LANGKAH 4 : (XᵀWX)^(-1)")
print("="*70)

display(

    pd.DataFrame(

        XTWX_inv,

        index=column_names,

        columns=column_names

    )

)

# ==========================================================
# LANGKAH 5
# Hitung XᵀWy
# ==========================================================

XTWy = XTW @ y

print()
print("="*70)
print("LANGKAH 5 : XᵀWy")
print("="*70)

display(

    pd.DataFrame(

        XTWy,

        index=column_names,

        columns=["Value"]

    )

)

# ==========================================================
# LANGKAH 6
# Hitung β
# ==========================================================

beta = XTWX_inv @ XTWy

print()
print("="*70)
print("LANGKAH 6 : β")
print("="*70)

beta_df = pd.DataFrame(

    {

        "Koefisien": beta

    },

    index=column_names

)

display(beta_df)

print()

print_success(
    "Koefisien regresi berhasil dihitung."
)


MENGHITUNG KOEFISIEN REGRESI LIME

LANGKAH 1 : Xᵀ


,0,1,2,3
Intercept,1.0,1.0,1.0,1.0
aplikasi,0.0,0.0,1.0,1.0
guna,0.0,1.0,0.0,1.0



LANGKAH 2 : XᵀW


,0,1,2,3
Intercept,0.727916,0.941893,0.941893,1.0
aplikasi,0.000000,0.000000,0.941893,1.0
guna,0.000000,0.941893,0.000000,1.0



LANGKAH 3 : XᵀWX


...


LANGKAH 5 : XᵀWy


,Value
Intercept,3.521368
aplikasi,1.930368
guna,1.892535



LANGKAH 6 : β


,Koefisien
Intercept,0.952154
aplikasi,0.041318
guna,0.001152



✅ Koefisien regresi berhasil dihitung.


In [ ]:
# =====================================================
# CELL 63 : VERIFIKASI DENGAN HASIL LIME
# =====================================================

print_header("VERIFIKASI HASIL REGRESI DENGAN LIME")

# ==========================================================
# LANGKAH 1
# Ambil explanation
# ==========================================================

explanation = sample_data["lime_explanation"]

# ==========================================================
# LANGKAH 2
# Ambil bobot kelas yang diprediksi
# ==========================================================

lime_weight = explanation.local_exp[predicted_label]

print()

print("Bobot dari LIME")

for feature_id, weight in lime_weight:

    print(feature_id, weight)

print()

# ==========================================================
# LANGKAH 3
# Mapping Feature
# ==========================================================

print("Feature Mapping")

for idx, token in feature_map.items():

    print(idx, "->", token)

print()

print_success("Verifikasi selesai.")


VERIFIKASI HASIL REGRESI DENGAN LIME

Bobot dari LIME
1 0.6461650545884946
2 -0.31397573824508407
0 -0.12137715459560759

Feature Mapping
0 -> aplikasi
1 -> guna

✅ Verifikasi selesai.


In [ ]:
print(predicted_label)
print(probabilities)

0
[[0.95072705 0.01089663 0.0383763 ]
 [0.95440847 0.00789986 0.03769178]
 [0.9945749  0.00292996 0.00249512]
 [0.99358517 0.00229748 0.00411741]]


In [ ]:
# =====================================================
# CELL 64 : CEK STRUKTUR INTERNAL LIME
# =====================================================

print_header("CEK STRUKTUR INTERNAL LIME")

exp = sample_data["lime_explanation"]

print(type(exp))

print()

print("="*70)
print("ATTRIBUTE")
print("="*70)

for x in dir(exp):

    if not x.startswith("_"):

        print(x)


CEK STRUKTUR INTERNAL LIME
<class 'lime.explanation.Explanation'>

ATTRIBUTE
as_html
as_list
as_map
as_pyplot_figure
available_labels
class_names
domain_mapper
intercept
local_exp
local_pred
mode
predict_proba
random_state
save_to_file
score
show_in_notebook
top_labels


modifikasi

In [ ]:
# =====================================================
# CELL 24A : CEK LOKASI FILE LIME
# =====================================================

import lime
import inspect

print(inspect.getfile(lime))

/usr/local/lib/python3.12/dist-packages/lime/__init__.py


In [ ]:
import inspect
import lime.lime_base

print(inspect.getfile(lime.lime_base))

/usr/local/lib/python3.12/dist-packages/lime/lime_base.py


In [ ]:
import inspect
import lime.lime_base

print(inspect.getsource(lime.lime_base.LimeBase.explain_instance_with_data))

    def explain_instance_with_data(self,
                                   neighborhood_data,
                                   neighborhood_labels,
                                   distances,
                                   label,
                                   num_features,
                                   feature_selection='auto',
                                   model_regressor=None):
        """Takes perturbed data, labels and distances, returns explanation.

        Args:
            neighborhood_data: perturbed data, 2d array. first element is
                               assumed to be the original data point.
            neighborhood_labels: corresponding perturbed labels. should have as
                                 many columns as the number of possible labels.
            distances: distances to original data point.
            label: label for which we want an explanation
            num_features: maximum number of features in explanation
            fea

X
T
X
T
W
X
T
WX
(X
T
WX)
−1
X
T
Wy
β=(X
T
WX)
−1
X
T
Wy

Perhitungan manual dilakukan dengan menggunakan bentuk matematis regresi berbobot pada LIME. Jumlah perturbasi pada perhitungan manual disederhanakan agar seluruh proses pembentukan matriks, pembobotan kernel, dan estimasi koefisien regresi dapat ditunjukkan secara eksplisit. Implementasi LIME pada library sebenarnya menghasilkan ribuan perturbasi secara acak, sehingga nilai koefisien dapat berbeda secara numerik, namun proses matematis yang digunakan tetap mengikuti persamaan regresi berbobot yang sama.